## Libraries

In [1]:
import pandas as pd
import numpy as np

In [ ]:
def extract_group_survival(df):
    """
    Создает признак групповой выживаемости на основе фамилий и номеров билетов.
    """
    # 1. Извлекаем фамилию
    df['Last_Name'] = df['Name'].apply(lambda x: x.split(',')[0])
    
    # 2. Группируем по фамилии и цене билета (семьи)
    # 0.5 — нейтральное значение (неизвестно)
    DEFAULT_SURVIVAL_VALUE = 0.5
    df['Group_Survival'] = DEFAULT_SURVIVAL_VALUE

    # Группировка по фамилии и цене билета
    for _, grp_df in df.groupby(['Last_Name', 'Fare']):
        if len(grp_df) > 1:
            for ind, row in grp_df.iterrows():
                smax = grp_df.drop(ind)['Survived'].max()
                smin = grp_df.drop(ind)['Survived'].min()
                if smax == 1.0:
                    df.loc[ind, 'Group_Survival'] = 1
                elif smin == 0.0:
                    df.loc[ind, 'Group_Survival'] = 0

    # 3. Дополняем проверкой по номеру билета (друзья/компании)
    for _, grp_df in df.groupby('Ticket'):
        if len(grp_df) > 1:
            for ind, row in grp_df.iterrows():
                if (df.loc[ind, 'Group_Survival'] == 0.5):
                    smax = grp_df.drop(ind)['Survived'].max()
                    smin = grp_df.drop(ind)['Survived'].min()
                    if smax == 1.0:
                        df.loc[ind, 'Group_Survival'] = 1
                    elif smin == 0.0:
                        df.loc[ind, 'Group_Survival'] = 0
                        
    return df


In [4]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

In [5]:
train_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [6]:
test_df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [7]:
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

full_df = extract_group_survival(full_df)

print("Unique values of Group_Survival:", full_df['Group_Survival'].unique())
full_df[['Name', 'Ticket', 'Group_Survival']].head(10)

Unique values of Group_Survival: [0.5 0.  1. ]


,Name,Ticket,Group_Survival
0,"Braund, Mr. Owen Harris",A/5 21171,0.5
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",PC 17599,0.5
2,"Heikkinen, Miss. Laina",STON/O2. 3101282,0.5
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803,0.0
4,"Allen, Mr. William Henry",373450,0.5
5,"Moran, Mr. James",330877,0.5
6,"McCarthy, Mr. Timothy J",17463,0.5
7,"Palsson, Master. Gosta Leonard",349909,0.0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",347742,1.0
9,"Nasser, Mrs. Nicholas (Adele Achem)",237736,0.0


In [8]:
# Extract Title from Name
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles into more common ones
mapping = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 'Major': 'Mr', 
           'Col': 'Mr', 'Dr': 'Mr', 'Rev': 'Mr', 'Capt': 'Mr', 
           'Jonkheer': 'Mr', 'Sir': 'Mr', 'Lady': 'Mrs', 
           'Don': 'Mr', 'Countess': 'Mrs', 'Dona': 'Mrs'}
full_df.replace({'Title': mapping}, inplace=True)

# Impute missing Age with median age of each Title
full_df['Age'] = full_df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

print("Unique Titles found:", full_df['Title'].unique())
print("Missing values in Age after imputation:", full_df['Age'].isnull().sum())

Unique Titles found: <StringArray>
['Mr', 'Mrs', 'Miss', 'Master']
Length: 4, dtype: str
Missing values in Age after imputation: 0


In [9]:
full_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Last_Name,Group_Survival,Title
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund,0.5,Mr
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings,0.5,Mrs
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen,0.5,Miss
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle,0.0,Mrs
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Allen,0.5,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,NaN,3,"Spector, Mr. Woolf",male,30.0,0,0,A.5. 3236,8.0500,NaN,S,Spector,0.5,Mr
1305,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,Oliva y Ocana,1.0,Mrs
1306,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,Saether,0.5,Mr
1307,1308,NaN,3,"Ware, Mr. Frederick",male,30.0,0,0,359309,8.0500,NaN,S,Ware,0.5,Mr


In [10]:
full_df.isna().sum()

PassengerId          0
Survived           418
Pclass               0
Name                 0
Sex                  0
Age                  0
SibSp                0
Parch                0
Ticket               0
Fare                 1
Cabin             1014
Embarked             2
Last_Name            0
Group_Survival       0
Title                0
dtype: int64

In [13]:
full_df['Fare'] = full_df['Fare'].fillna(full_df['Fare'].median())
full_df['Embarked'] = full_df['Embarked'].fillna(full_df['Embarked'].mode()[0])

In [14]:
full_df.isna().sum()

PassengerId          0
Survived           418
Pclass               0
Name                 0
Sex                  0
Age                  0
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin             1014
Embarked             0
Last_Name            0
Group_Survival       0
Title                0
dtype: int64

In [15]:
full_df['Sex'] = full_df['Sex'].map({'male': 0, 'female': 1})

In [16]:
full_df = pd.get_dummies(full_df, columns=['Embarked', 'Title'], prefix=['Emb', 'Title'])

In [17]:
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Last_Name', 'Survived']
X_full = full_df.drop(columns=drop_cols)
y_full = full_df['Survived']

In [18]:
X_train = X_full[:891] # 891 - размер оригинального train
y_train = y_full[:891].astype(int)
X_test = X_full[891:]

In [19]:
print("Final features for training:", X_train.columns.tolist())
print(f"Missing values in X_train: {X_train.isna().sum().sum()}")

Final features for training: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Group_Survival', 'Emb_C', 'Emb_Q', 'Emb_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs']
Missing values in X_train: 0
